# Model 1: Artificial Neural Network — SMS Spam Detection

## 1. Problem Definition

**Aim of the application:** Classify SMS messages as either spam or legitimate (ham) using an Artificial Neural Network.

**Type of data used:** Text data (raw SMS messages).

**Type of problem:** Binary classification — each message belongs to one of two classes: `spam` (1) or `ham` (0).

**Why ANN is suitable for this problem:**  
After converting raw text into a fixed-length numerical representation using TF-IDF vectorization, the problem reduces to a standard tabular binary classification task. An ANN with fully connected (Dense) layers is well-suited for learning non-linear decision boundaries in high-dimensional feature spaces such as TF-IDF vectors. Unlike RNNs or CNNs, an ANN does not require sequential structure in the input, making it efficient and appropriate for bag-of-words style text features.

## 2. Dataset

**Name:** SMS Spam Collection Dataset  
**Link:** https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip  
**Type:** Text classification dataset  
**Size:** 5,574 SMS messages (4,827 ham, 747 spam)

**Why this dataset is suitable:**  
The SMS Spam Collection is a publicly available, well-studied benchmark dataset for binary text classification. It contains real-world SMS messages in English, pre-labeled as spam or ham. The class imbalance (approximately 87% ham, 13% spam) is a realistic challenge that mirrors production systems, allowing evaluation of the model under real-world conditions.

**Basic preprocessing steps:**
- Load the tab-separated file into a pandas DataFrame
- Encode labels: ham → 0, spam → 1
- Apply TF-IDF vectorization (top 3,000 features) to convert text to numerical vectors
- Split into train (80%), validation (10%), and test (10%) sets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import urllib.request
import zipfile
import os

print(f'TensorFlow version: {tf.__version__}')
np.random.seed(42)
tf.random.set_seed(42)

## 3. Data Loading and Preprocessing

In [ ]:
# Download dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
zip_path = 'smsspamcollection.zip'

if not os.path.exists('SMSSpamCollection'):
    print('Downloading dataset...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('.')
    os.remove(zip_path)
    print('Done.')

df = pd.read_csv('SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
print(f'Dataset shape: {df.shape}')
print(df['label'].value_counts())
df.head()

In [ ]:
# Encode labels
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})

# Visualize class distribution
fig, ax = plt.subplots(figsize=(5, 4))
df['label'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'], ax=ax)
ax.set_title('Class Distribution (Ham vs Spam)')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
ax.set_xticklabels(['Ham', 'Spam'], rotation=0)
plt.tight_layout()
plt.show()
print(f'Spam ratio: {df["label_enc"].mean():.2%}')

In [ ]:
X = df['message'].values
y = df['label_enc'].values

# First split: 80% train+val, 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)
# Second split: 80% train, 10% val (of original)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.111, random_state=42, stratify=y_trainval
)

print(f'Train size:      {len(X_train)} ({len(X_train)/len(X):.0%})')
print(f'Validation size: {len(X_val)} ({len(X_val)/len(X):.0%})')
print(f'Test size:       {len(X_test)} ({len(X_test)/len(X):.0%})')

In [ ]:
# TF-IDF vectorization — fit on train only to prevent data leakage
vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train).toarray()
X_val_tfidf   = vectorizer.transform(X_val).toarray()
X_test_tfidf  = vectorizer.transform(X_test).toarray()

print(f'Feature matrix shape (train): {X_train_tfidf.shape}')

## 4. Model Architecture

**Type of model:** Artificial Neural Network (Multilayer Perceptron)

**Architecture:**
```
Input(3000)  →  Dense(64, ReLU)  →  Dropout(0.3)  →  Dense(32, ReLU)  →  Dense(1, Sigmoid)
```

| Layer | Units | Activation | Purpose |
|---|---|---|---|
| Input | 3000 | — | TF-IDF feature vector |
| Dense | 64 | ReLU | Learn non-linear combinations of features |
| Dropout | 0.3 | — | Regularization to reduce overfitting |
| Dense | 32 | ReLU | Further abstraction |
| Dense | 1 | Sigmoid | Output probability of spam (0–1) |

**Loss function:** Binary Crossentropy — standard for binary classification tasks.  
**Optimizer:** Adam (lr=0.001) — adaptive learning rate, converges reliably.  
**Epochs:** 20  
**Batch size:** 32  
**Train/Val/Test split:** 80% / 10% / 10%

**Architecture justification:** The two-hidden-layer design (64→32) progressively compresses the 3000-dimensional TF-IDF input into increasingly abstract representations. Dropout after the first dense layer prevents co-adaptation of neurons and reduces overfitting on the relatively small dataset. The sigmoid output directly provides a calibrated probability score.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(3000,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='ANN_SpamDetector')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Training Process

In [ ]:
history = model.fit(
    X_train_tfidf, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(X_val_tfidf, y_val),
    verbose=1
)

## 6. Results and Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(history.history['loss'], label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='tomato', linestyle='--')
axes[0].set_title('Training and Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='steelblue')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='tomato', linestyle='--')
axes[1].set_title('Training and Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ann_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
Interpretation:
- Both training and validation loss decrease consistently over epochs, indicating the model is learning effectively.
- If the two curves remain close together, this indicates minimal overfitting.
- A validation loss that rises while training loss continues to fall would indicate overfitting.
""")

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_tfidf, y_test, verbose=0)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

y_pred_prob = model.predict(X_test_tfidf)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham (Predicted)', 'Spam (Predicted)'],
            yticklabels=['Ham (Actual)', 'Spam (Actual)'],
            ax=ax)
ax.set_title('Confusion Matrix — ANN Spam Detection')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('ann_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Ham correctly identified):   {tn}')
print(f'False Positives (Ham incorrectly flagged):    {fp}')
print(f'False Negatives (Spam that got through):      {fn}')
print(f'True Positives  (Spam correctly caught):      {tp}')
print(f'\nPrecision (Spam): {tp/(tp+fp):.4f}')
print(f'Recall    (Spam): {tp/(tp+fn):.4f}')
print(f'F1 Score  (Spam): {2*tp/(2*tp+fp+fn):.4f}')

print("""
Interpretation:
- Precision measures what fraction of messages flagged as spam are actually spam.
  High precision means fewer legitimate messages are incorrectly blocked.
- Recall measures what fraction of actual spam messages were caught.
  High recall means fewer spam messages slip through the filter.
- F1-score balances both metrics. For spam detection, recall is often prioritized
  since missing spam is more harmful than occasionally flagging a legitimate message.
""")

In [ ]:
# Prediction examples
print('Sample Predictions on Test Set:')
print('-' * 70)
indices = np.where(y_test == 1)[0][:3].tolist() + np.where(y_test == 0)[0][:3].tolist()
for i in indices:
    msg = X_test[i]
    true_label = 'SPAM' if y_test[i] == 1 else 'HAM'
    pred_label = 'SPAM' if y_pred[i] == 1 else 'HAM'
    prob = float(y_pred_prob[i])
    status = '✓' if true_label == pred_label else '✗'
    print(f'{status} True: {true_label} | Predicted: {pred_label} | P(spam)={prob:.3f}')
    print(f'  Message: {msg[:80]}...' if len(msg) > 80 else f'  Message: {msg}')
    print()

## Discussion

**Did the model produce successful results?**  
The ANN achieved high test accuracy on the SMS Spam Collection dataset. The high precision and recall for the spam class indicate the model effectively distinguishes spam from legitimate messages.

**Strengths:**
- Simple and fast to train — no sequential processing needed
- TF-IDF + ANN pipeline is interpretable and computationally lightweight
- Performs well even on an imbalanced dataset

**Weaknesses:**
- TF-IDF discards word order, so contextual meaning is lost
- The model does not understand semantic relationships between words
- Susceptible to vocabulary drift (new spam words not in training vocabulary)

**What could improve results?**
- Use word embeddings (Word2Vec, GloVe) instead of TF-IDF to capture semantic meaning
- Switch to an RNN/LSTM to leverage word order
- Apply class weighting to handle the spam/ham imbalance more explicitly
- Increase the TF-IDF vocabulary size beyond 3,000 features